# beat_this sobre GuitarSet

Corre el detector de pulso beat_this (CPJKU, licencia MIT) sobre los 144 fragmentos de bossa nova y cantautor de GuitarSet, y devuelve los pulsos y los inicios de compás en `beat-this-guitarset.zip` para medirlos en la Mac con `bench.py` contra el detector de hoy.

Antes de ejecutar: menú Entorno de ejecución > Cambiar tipo de entorno de ejecución > GPU T4. Después, Entorno de ejecución > Ejecutar todas. Al final se descarga el zip solo.

In [ ]:
import torch
assert torch.cuda.is_available(), "No hay GPU: cambia el tipo de entorno de ejecución a T4 y vuelve a ejecutar"
print(torch.cuda.get_device_name(0))

In [ ]:
!pip -q install "git+https://github.com/CPJKU/beat_this.git" einops rotary-embedding-torch soxr

In [ ]:
# Sólo el audio del micrófono: 657 MB.
!wget -q -O audio_mono-mic.zip "https://zenodo.org/api/records/3371780/files/audio_mono-mic.zip/content"
!unzip -q -o audio_mono-mic.zip -d audio && rm audio_mono-mic.zip
import glob
wavs = sorted(w for w in glob.glob("audio/*_mic.wav") if w.split("/")[-1].split("_")[1][:2] in ("BN", "SS"))
print(len(wavs), "fragmentos")

In [ ]:
from beat_this.inference import File2Beats
import csv, os, time

# Sin DBN: el modelo ya da los pulsos limpios, y el DBN fuerza un tempo constante que
# en grabaciones tocadas a mano no existe.
file2beats = File2Beats(checkpoint_path="final0", device="cuda", dbn=False)
t0 = time.time()
with open("beat-this.csv", "w", newline="") as f:
    out = csv.writer(f)
    out.writerow(["file", "kind", "time"])
    for i, wav in enumerate(wavs, 1):
        stem = os.path.basename(wav)[: -len("_mic.wav")]
        beats, downbeats = file2beats(wav)
        out.writerows([stem, "beat", round(float(t), 4)] for t in beats)
        out.writerows([stem, "downbeat", round(float(t), 4)] for t in downbeats)
        if i % 24 == 0:
            print(i, "/", len(wavs))
print("listo en", round(time.time() - t0), "s")

In [ ]:
!zip -q beat-this-guitarset.zip beat-this.csv
from google.colab import files
files.download("beat-this-guitarset.zip")